# 06 - Pairwise judging with Kaggle models (Task 7) + report assets (Task 8)
**Create this as a Kaggle *Benchmark* notebook** (kaggle.com/benchmarks -> create a task) so `kaggle_benchmarks` and the model proxy are available. **Accelerator:** none needed · **Inputs:** outputs of notebooks 02, 03a, 03b, 03c, 04, 05 · **Runtime:** ~2-3 h (about 570 judge requests at >=5 s spacing)

This notebook only *judges saved pairwise files*; it never regenerates model outputs. Judging is resumable: if the session dies or the judge quota runs out, run it again - completed pairs are reused. Save a version when done.

In [ ]:
# ==== EDIT THIS: your GitHub fork of the repository ====
REPO_URL = "https://github.com/Malik9102/small-model-alignment.git"
REPO_DIR = "/kaggle/working/small-model-alignment"

import glob, os, shutil, subprocess, sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"] = "1"   # live script output in the Kaggle log
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"], check=False)

def sh(command):
    """Run a shell command, stream its output, and STOP the notebook if it fails.
    (A plain `!command` keeps going after an error, so a crashed training run would go unnoticed.)"""
    print(f"$ {command}", flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"Command failed with exit code {process.returncode}: {command}")

def find_input(pattern):
    """Locate a file/dir inside attached Kaggle inputs (any dataset/notebook slug)."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"Nothing matching {pattern!r} under /kaggle/input - did you attach the right notebook output?")
    if len(hits) > 1:
        print(f"[find_input] {len(hits)} matches for {pattern!r}, using {hits[0]}")
    return hits[0]

In [ ]:
sh(f"""python -m pip install -q pyyaml matplotlib""")

In [ ]:
import kaggle_benchmarks as kbench
print('models visible to kaggle_benchmarks:', sorted(kbench.llms))
JUDGE = 'google/gemini-3.6-flash'   # if not listed above, pick a listed model and record it in the report

In [ ]:
# Gather everything from the attached inputs (metrics + generations only - no checkpoints needed here)
for sub, marker in (('generations', 'sft_alpaca.json'), ('eval', 'dpo_uf_vs_sft_pairs.jsonl')):
    src = os.path.dirname(find_input(f'outputs/{sub}/{marker}'))
    shutil.copytree(src, f'outputs/{sub}', dirs_exist_ok=True)
extra_generations = ['lora_persona', 'smollm_base_persona', 'lora_persona_beta05', 'lora_persona_1epoch',
                     'dpo_hh_lr5e6_hh', 'dpo_hh_lr5e6_alpaca']
for stem in extra_generations:
    for ext in ('.json', '.meta.json'):
        try: shutil.copy(find_input(f'outputs/generations/{stem}{ext}'), 'outputs/generations/')
        except FileNotFoundError as e: print('[missing]', e)
shutil.copy(find_input('outputs/eval/lora_vs_base_persona_pairs.jsonl'), 'outputs/eval/')  # built in notebook 04
for run in ('sft', 'dpo_hh', 'dpo_uf', 'dpo_hh_lr5e6', 'lora_persona', 'lora_persona_beta05', 'lora_persona_1epoch'):
    try: src = os.path.dirname(find_input(f'outputs/{run}/summary.json'))
    except FileNotFoundError as e: print('[missing]', e); continue
    os.makedirs(f'outputs/{run}', exist_ok=True)
    for name in ('summary.json', 'config_used.json', 'metrics.jsonl'):
        if os.path.exists(os.path.join(src, name)): shutil.copy(os.path.join(src, name), f'outputs/{run}/')
sh(f"""ls outputs/*""")

## Extra pairwise files: ablations, cross-set (alignment tax), SFT vs base, full 300-pair position swaps
Written to disk *before* judging.

In [ ]:
G = 'outputs/generations'
EXTRA_PAIRS = [
    # (candidate generations, baseline generations, pairs file stem)
    ('dpo_hh_lr5e6_hh',         'sft_hh',              'dpo_hh_lr5e6_vs_sft'),
    ('dpo_hh_lr5e6_alpaca',     'sft_alpaca',          'dpo_hh_lr5e6_vs_sft_alpaca'),
    ('dpo_hh_alpaca',           'sft_alpaca',          'dpo_hh_vs_sft_alpaca'),
    ('dpo_uf_hh',               'sft_hh',              'dpo_uf_vs_sft_hh'),
    ('sft_alpaca',              'base_alpaca',         'sft_vs_base_alpaca'),
    ('lora_persona_beta05',     'smollm_base_persona', 'lora_beta05_vs_base_persona'),
    ('lora_persona_1epoch',     'smollm_base_persona', 'lora_1epoch_vs_base_persona'),
]
for cand, base, stem in EXTRA_PAIRS:
    if os.path.exists(f'{G}/{cand}.json') and os.path.exists(f'{G}/{base}.json'):
        sh(f"python scripts/build_pairs.py --candidate {G}/{cand}.json --baseline {G}/{base}.json --output outputs/eval/{stem}_pairs.jsonl")
    else:
        print('[skip]', stem, '- missing generations')
# full 300-pair A/B-swapped copies (notebook 05 made 100-pair ones)
for stem in ('dpo_uf_vs_sft', 'dpo_hh_vs_sft'):
    sh(f"python scripts/build_pairs.py --mirror_of outputs/eval/{stem}_pairs.jsonl --output outputs/eval/{stem}_mirrored_pairs.jsonl")

## Judge (5 pairs/request, >=5 s between attempts, 3 attempts per batch, failures logged, resumable)

In [ ]:
from alignment.judging import run_judging
# (pairs file stem, rubric). Ordered by priority: if the judge quota runs out, rerun this cell later -
# finished pairs are reused, so it resumes where it stopped.
JOBS = [
    # --- required DPO-vs-SFT comparisons (manual 12-13) ---
    ('dpo_uf_vs_sft',                'general'),
    ('dpo_hh_vs_sft',                'safety'),
    # --- LoRA persona: style and quality/safety judged separately ---
    ('lora_vs_base_persona',         'persona_style'),
    ('lora_vs_base_persona',         'persona_quality'),
    # --- ablations ---
    ('dpo_hh_lr5e6_vs_sft',          'safety'),
    ('lora_beta05_vs_base_persona',  'persona_style'),
    ('lora_beta05_vs_base_persona',  'persona_quality'),
    ('lora_1epoch_vs_base_persona',  'persona_style'),
    ('lora_1epoch_vs_base_persona',  'persona_quality'),
    # --- cross-set (alignment tax) and SFT effect ---
    ('dpo_hh_vs_sft_alpaca',         'general'),
    ('dpo_hh_lr5e6_vs_sft_alpaca',   'general'),
    ('dpo_uf_vs_sft_hh',             'safety'),
    ('sft_vs_base_alpaca',           'general'),
    # --- judge position-bias checks: the same 300 pairs with A/B swapped ---
    ('dpo_uf_vs_sft_mirrored',       'general'),
    ('dpo_hh_vs_sft_mirrored',       'safety'),
]
for name, rubric in JOBS:
    suffix = '' if rubric in ('general', 'safety') else f'_{rubric}'
    pairs_path = f'outputs/eval/{name}_pairs.jsonl'
    if not os.path.exists(pairs_path):
        print('[skip] missing', pairs_path); continue
    run_judging(
        pairs_path=pairs_path,
        output_path=f'outputs/eval/{name}{suffix}_judged.jsonl',
        judge_model=JUDGE, batch_size=5, sleep_seconds=5, max_retries=3,
        reuse_judgments=True, rubric=rubric, backend='kbench',
    )

## Summaries with 95% confidence intervals and position-swap consistency

In [ ]:
S = [
    ('UF-DPO vs SFT (AlpacaEval)',            'dpo_uf_vs_sft_judged',                          'dpo_uf_vs_sft_mirrored_judged'),
    ('HH-DPO vs SFT (HH set)',                'dpo_hh_vs_sft_judged',                          'dpo_hh_vs_sft_mirrored_judged'),
    ('HH-DPO lr5e-6 vs SFT (HH set)',         'dpo_hh_lr5e6_vs_sft_judged',                    None),
    ('HH-DPO vs SFT (AlpacaEval)',            'dpo_hh_vs_sft_alpaca_judged',                   None),
    ('HH-DPO lr5e-6 vs SFT (AlpacaEval)',     'dpo_hh_lr5e6_vs_sft_alpaca_judged',             None),
    ('UF-DPO vs SFT (HH set)',                'dpo_uf_vs_sft_hh_judged',                       None),
    ('SFT vs base (AlpacaEval)',              'sft_vs_base_alpaca_judged',                     None),
    ('LoRA vs base: style',                   'lora_vs_base_persona_persona_style_judged',     None),
    ('LoRA vs base: quality/safety',          'lora_vs_base_persona_persona_quality_judged',   None),
    ('LoRA beta0.5 vs base: style',           'lora_beta05_vs_base_persona_persona_style_judged',   None),
    ('LoRA beta0.5 vs base: quality/safety',  'lora_beta05_vs_base_persona_persona_quality_judged', None),
    ('LoRA 1 epoch vs base: style',           'lora_1epoch_vs_base_persona_persona_style_judged',   None),
    ('LoRA 1 epoch vs base: quality/safety',  'lora_1epoch_vs_base_persona_persona_quality_judged', None),
]
for title, judged, mirrored in S:
    path = f'outputs/eval/{judged}.jsonl'
    if not os.path.exists(path):
        print('[skip]', title); continue
    cmd = f"python scripts/summarize_judgments.py --judged {path} --name '{title}'"
    if mirrored and os.path.exists(f'outputs/eval/{mirrored}.jsonl'):
        cmd += f" --mirror_judged outputs/eval/{mirrored}.jsonl"
    sh(cmd)
    print()

## Does the judge prefer longer answers? (length-bias check)

In [ ]:
E = 'outputs/eval'
comparisons = [
    ('UF-DPO vs SFT (AlpacaEval)', 'dpo_uf_vs_sft', 'dpo_uf_vs_sft'),
    ('HH-DPO vs SFT (HH set)', 'dpo_hh_vs_sft', 'dpo_hh_vs_sft'),
    ('HH-DPO lr5e-6 vs SFT (HH set)', 'dpo_hh_lr5e6_vs_sft', 'dpo_hh_lr5e6_vs_sft'),
    ('HH-DPO vs SFT (AlpacaEval)', 'dpo_hh_vs_sft_alpaca', 'dpo_hh_vs_sft_alpaca'),
    ('SFT vs base (AlpacaEval)', 'sft_vs_base_alpaca', 'sft_vs_base_alpaca'),
]
args = [(n, f'{E}/{p}_pairs.jsonl', f'{E}/{j}_judged.jsonl') for n, p, j in comparisons if os.path.exists(f'{E}/{j}_judged.jsonl')]
names = ' '.join(f"'{n}'" for n, _, _ in args)
sh(f"python scripts/analyze_length_bias.py --pairs {' '.join(p for _, p, _ in args)} --judged {' '.join(j for _, _, j in args)} --name {names} --output {E}/length_bias.json")

## Report assets: plots, tables, examples, generation statistics

In [ ]:
sh(f"""python scripts/plot_metrics.py --sft outputs/sft --dpo outputs/dpo_hh outputs/dpo_uf outputs/dpo_hh_lr5e6 --lora outputs/lora_persona outputs/lora_persona_beta05 outputs/lora_persona_1epoch --overlay_dpo outputs/dpo_hh outputs/dpo_hh_lr5e6 --overlay_lora outputs/lora_persona outputs/lora_persona_beta05 outputs/lora_persona_1epoch --output_dir outputs/plots""")
sh(f"""python scripts/collect_results.py --runs_root outputs --output outputs/report_tables.md""")

In [ ]:
EX = [('dpo_uf_vs_sft', 'dpo_uf_vs_sft', 3), ('dpo_hh_vs_sft', 'dpo_hh_vs_sft', 3), ('dpo_hh_lr5e6_vs_sft', 'dpo_hh_lr5e6_vs_sft', 3),
      ('lora_vs_base_persona', 'lora_vs_base_persona_persona_style', 2), ('lora_vs_base_persona', 'lora_vs_base_persona_persona_quality', 2)]
for pairs, judged, k in EX:
    if os.path.exists(f'outputs/eval/{judged}_judged.jsonl'):
        sh(f'python scripts/show_examples.py --pairs outputs/eval/{pairs}_pairs.jsonl --judged outputs/eval/{judged}_judged.jsonl --k {k} --output outputs/eval/examples_{judged}.md > /dev/null')
sh(f"""python scripts/analyze_generations.py outputs/generations/*.json --output outputs/eval/generation_stats.json""")

In [ ]:
sh(f"""cd {REPO_DIR} && zip -qr /kaggle/working/06_final_results.zip outputs && ls -lh /kaggle/working/06_final_results.zip""")